#### Anthropic Building with Claude API 
#### Lesson 01
- Modified to include learning from my language project
- https://anthropic.skilljar.com/claude-with-the-anthropic-api/296766


In [1]:
import json
import logging
import os
import sys

from dotenv import load_dotenv
load_dotenv()


from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("CLAUDE_COURSE_API_KEY"))
model = "claude-sonnet-4-0"



#### Demo to highlight how LLMs work
- Claude doesn't store convo history
    - If you ask a question and a follow-up question, it won't remember
    - To get it to remember, create a series of functions to add Claude's response to the convo history/
- Parameterize system prompts in a dict
- Customize a system prompt focused on language learning

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": 0.1
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Define a system prompt to guide the assistant's behavior
# TODO: Put this in a config file
system_prompt = """
You are professor of languages focused on English, Italian and French. 
You are helpful, precise, and concise. 
You are an expert at differentiating between the nuances of these languages and providing accurate translations.
You also have a background in history and can provide real-world examples of historical events to illustrate your examples when relevant.
"""

In [4]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Translate musical theater in both Italian and French")

# Get Claude's response
answer = chat(messages, system=system_prompt)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
# This will highlight Claude's ability to maintain history and context across multiple chat interactions and build on previous responses.
add_user_message(messages, "Write a 4 word sentence using the Italian translation and a 4 word sentence using the French translation")

# Get the follow-up response with full context
final_answer = chat(messages, system=system_prompt)

In [5]:
print(answer)

**Italian:** teatro musicale

**French:** théâtre musical

Both translations are direct and commonly used in their respective languages. 

**Note:** In Italian, you might also encounter "musical" (borrowed from English) in casual conversation, but "teatro musicale" is the proper Italian term. In French, "comédie musicale" is also frequently used, particularly when referring to Broadway-style musicals.


In [6]:
print(final_answer)

**Italian:** Il teatro musicale piace.
*(Musical theater is enjoyable.)*

**French:** Le théâtre musical plaît.
*(Musical theater is pleasing.)*

Both sentences use similar structures with the verb "piacere/plaire" (to please/be pleasing), which is a common way to express enjoyment or appreciation in both languages.


#### Streaming 
- The above simple set-up works for learning, but in the real world, this set-up is slow.
- For an app or production, we would want to utilize the advantages of streaming
- "When you enable streaming, Claude sends back several types of events:

    - MessageStart - A new message is being sent
    - ContentBlockStart - Start of a new block containing text, tool use, or other content
    - ContentBlockDelta - Chunks of the actual generated text
    - ContentBlockStop - The current content block has been completed
    - MessageDelta - The current message is complete
    - MessageStop - End of information about the current message"

In [9]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

In [10]:
final_message

ParsedMessage(id='msg_01KomZWo5LoYXBmiY9JoogH2', container=None, content=[ParsedTextBlock(citations=None, text='**Italian:** Il teatro musicale affascina. \n(Musical theater fascinates.)\n\n**French:** Le théâtre musical plaît.\n(Musical theater pleases.)', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=132, output_tokens=42, server_tool_use=None, service_tier='standard'))

##### TODO: Parse message using JSON package